# EDA

## Subreddit-1: learnprogramming

## Part-1: Data Overview

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = '/Users/yangshuting/Desktop/UChicago/courses/spring_2026/MACS_30200_Research_Design/final_proposal/data/learnprogramming/'
POSTS_PATH    = DATA_DIR + 'r_learnprogramming_posts.jsonl'
COMMENTS_PATH = DATA_DIR + 'r_learnprogramming_comments.jsonl'
OBS_END = pd.Timestamp('2026-01-01', tz='UTC')

In [ ]:
# load posts - keep only needed columns
POSTS_COLS = ['author', 'created_utc', 'id', 'title', 'selftext', 'score', 'num_comments']

posts = []
with open(POSTS_PATH) as f:
    for line in f:
        d = json.loads(line)
        posts.append({k: d.get(k) for k in POSTS_COLS})

posts_df = pd.DataFrame(posts)
posts_df.shape

In [ ]:
# load comments - keep only needed columns
COMMENTS_COLS = ['author', 'created_utc', 'id', 'body', 'score', 'parent_id', 'link_id']

comments = []
with open(COMMENTS_PATH) as f:
    for line in f:
        d = json.loads(line)
        comments.append({k: d.get(k) for k in COMMENTS_COLS})

comments_df = pd.DataFrame(comments)
comments_df.shape

In [ ]:
# get an overview of posts
posts_df.head()

In [ ]:
# get an overview of comments
comments_df.head()

## Part-2: Data Cleaning

### 2.1 Missing values

In [ ]:
# check missing values
null_posts    = posts_df.isnull().sum()
null_comments = comments_df.isnull().sum()
print('=== Posts ===')
print(null_posts)
print('\n=== Comments ===')
print(null_comments)

In [ ]:
# drop columns that have at least 30% missing values
posts_df.drop(columns=posts_df.columns[null_posts > len(posts_df) * 0.3], inplace=True)
comments_df.drop(columns=comments_df.columns[null_comments > len(comments_df) * 0.3], inplace=True)

# check remaining columns
print('posts columns:', posts_df.columns.tolist())
print('comments columns:', comments_df.columns.tolist())

### 2.2 Remove deleted accounts

In [ ]:
# check [deleted] and [removed] author proportion
print('=== Posts ===')
print(f'total: {len(posts_df):,}')
print(f'[deleted]: {(posts_df["author"] == "[deleted]").sum():,}  ({(posts_df["author"] == "[deleted]").mean()*100:.1f}%)')
print(f'[removed]: {(posts_df["author"] == "[removed]").sum():,}')

print('\n=== Comments ===')
print(f'total: {len(comments_df):,}')
print(f'[deleted]: {(comments_df["author"] == "[deleted]").sum():,}  ({(comments_df["author"] == "[deleted]").mean()*100:.1f}%)')
print(f'[removed]: {(comments_df["author"] == "[removed]").sum():,}')

In [ ]:
# remove [deleted] and [removed] authors
posts_df    = posts_df[~posts_df['author'].isin(['[deleted]', '[removed]'])].reset_index(drop=True)
comments_df = comments_df[~comments_df['author'].isin(['[deleted]', '[removed]'])].reset_index(drop=True)

print(f'posts after filtering:    {len(posts_df):,}')
print(f'comments after filtering: {len(comments_df):,}')

### 2.3 Remove duplicates

In [ ]:
# check duplicate content: same user posting identical text
# posts: same author + same title; comments: same author + same body
dup_posts    = posts_df.duplicated(subset=['author', 'title'], keep=False).sum()
dup_comments = comments_df.duplicated(subset=['author', 'body'], keep=False).sum()

print(f'duplicate posts (same author + title):   {dup_posts:,}')
print(f'duplicate comments (same author + body): {dup_comments:,}')

In [ ]:
# drop duplicates, keep first occurrence
posts_df    = posts_df.drop_duplicates(subset=['author', 'title'], keep='first').reset_index(drop=True)
comments_df = comments_df.drop_duplicates(subset=['author', 'body'],  keep='first').reset_index(drop=True)

print(f'posts after dedup:    {len(posts_df):,}')
print(f'comments after dedup: {len(comments_df):,}')

### 2.4 Clean text fields

In [ ]:
# check [deleted] / [removed] text in content fields
print(f'posts with [deleted] selftext:  {(posts_df["selftext"] == "[deleted]").sum():,}')
print(f'posts with [removed] selftext:  {(posts_df["selftext"] == "[removed]").sum():,}')
print(f'comments with [deleted] body:   {(comments_df["body"] == "[deleted]").sum():,}')
print(f'comments with [removed] body:   {(comments_df["body"] == "[removed]").sum():,}')

In [ ]:
# replace [deleted] / [removed] text with NaN
posts_df['selftext'] = posts_df['selftext'].replace(['[deleted]', '[removed]'], None)
comments_df['body']  = comments_df['body'].replace(['[deleted]', '[removed]'], None)

# re-check missing after replacing
print(f'selftext missing after cleaning: {posts_df["selftext"].isnull().sum():,}')
print(f'body missing after cleaning:     {comments_df["body"].isnull().sum():,}')

### 2.5 Remove bots

In [ ]:
# check top 30 most active commenters to identify bots
top_commenters = comments_df['author'].value_counts().head(30)
top_commenters

In [ ]:
# define known bots and remove them
BOTS = [
    'AutoModerator', 'BotDefense', 'RepostSleuthBot', 'snapshillbot',
    'WikiTextBot', 'LinkifyBot', 'TweetPoster', 'imgurtranscriber',
    'MAGIC_EYE_BOT', 'RemindMeBot', 'haikusbot', 'converter-bot'
]

posts_df    = posts_df[~posts_df['author'].isin(BOTS)].reset_index(drop=True)
comments_df = comments_df[~comments_df['author'].isin(BOTS)].reset_index(drop=True)

print(f'posts after bot removal:    {len(posts_df):,}')
print(f'comments after bot removal: {len(comments_df):,}')

### 2.6 User activity records

In [ ]:
# convert timestamps
posts_df['created_utc']    = pd.to_datetime(posts_df['created_utc'], unit='s', utc=True)
comments_df['created_utc'] = pd.to_datetime(comments_df['created_utc'], unit='s', utc=True)

In [ ]:
# each post/comment = one transaction (following BG/NBD convention)
activity = pd.concat([
    posts_df[['author', 'created_utc']],
    comments_df[['author', 'created_utc']]
], ignore_index=True).sort_values('created_utc').reset_index(drop=True)

print(f'total activity records: {len(activity):,}')

In [ ]:
# compute per-user activity stats
user_stats = activity.groupby('author').agg(
    n_activity = ('created_utc', 'count'),
    first_post = ('created_utc', 'min'),
    last_post  = ('created_utc', 'max')
).reset_index()

print(f'total unique users: {len(user_stats):,}')
user_stats.head()

### 2.7 Filter low-activity users

In [ ]:
# compute the bottom 5% activity threshold (calculated per subreddit)
threshold = int(user_stats['n_activity'].quantile(0.05))
print(f'bottom 5% threshold: {threshold} activities')
print(f'users before filter: {len(user_stats):,}')

In [ ]:
# drop users below the threshold
user_stats = user_stats[user_stats['n_activity'] >= threshold].reset_index(drop=True)
print(f'users after filter: {len(user_stats):,}')

qualifying_users = set(user_stats['author'])
activity = activity[activity['author'].isin(qualifying_users)].reset_index(drop=True)

### 2.8 Summary

In [ ]:
# cleaning steps summary
summary = pd.DataFrame({
    'Step': [
        'Raw posts',
        'Raw comments',
        'After removing [deleted] / [removed] authors (posts)',
        'After removing [deleted] / [removed] authors (comments)',
        'After removing duplicates (posts)',
        'After removing duplicates (comments)',
        'After removing bots (posts)',
        'After removing bots (comments)',
        'Qualifying users'
    ],
    'Count': [
        640028,
        4464776,
        posts_df.shape[0],
        comments_df.shape[0],
        posts_df.shape[0],
        comments_df.shape[0],
        len(posts_df),
        len(comments_df),
        len(user_stats)
    ]
})
summary

## Part-3: User Activity

In [ ]:
# activity count per user distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(user_stats['n_activity'].clip(upper=500), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Activity Count per User (clipped at 500)', fontweight='bold')
axes[0].set_xlabel('Total Posts + Comments')
axes[0].set_ylabel('Number of Users')

axes[1].hist(np.log1p(user_stats['n_activity']), bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Activity Count per User (log scale)', fontweight='bold')
axes[1].set_xlabel('log(1 + activity count)')
axes[1].set_ylabel('Number of Users')

plt.tight_layout()
plt.show()

In [ ]:
# user tenure distribution
user_stats['tenure_days'] = (OBS_END - user_stats['first_post']).dt.days

plt.figure(figsize=(10, 4))
plt.hist(user_stats['tenure_days'], bins=50, color='steelblue', edgecolor='white')
plt.title('User Tenure Distribution (days to Jan 2026)', fontweight='bold')
plt.xlabel('Tenure (days)')
plt.ylabel('Number of Users')
plt.tight_layout()
plt.show()

In [ ]:
# summary stats
user_stats[['n_activity', 'tenure_days']].describe().round(1)

## Part-4: Temporal Patterns

In [ ]:
# monthly post volume over time
monthly_posts = posts_df.groupby(posts_df['created_utc'].dt.to_period('M')).size()

plt.figure(figsize=(15, 5))
monthly_posts.plot(color='steelblue', lw=1.5)
plt.title('Monthly Post Volume in r/learnprogramming (2009–2026)', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Number of Posts')
plt.tight_layout()
plt.show()

In [ ]:
# post volume by day of week and hour of day
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily  = posts_df['created_utc'].dt.day_name().value_counts().reindex(day_order)
hourly = posts_df['created_utc'].dt.hour.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].bar(daily.index, daily.values, color='steelblue', edgecolor='white')
axes[0].set_title('Posts by Day of Week', fontweight='bold')
axes[0].set_ylabel('Number of Posts')
axes[0].tick_params(axis='x', rotation=45)

axes[1].bar(hourly.index, hourly.values, color='steelblue', edgecolor='white')
axes[1].set_title('Posts by Hour of Day (UTC)', fontweight='bold')
axes[1].set_xlabel('Hour (UTC)')
axes[1].set_ylabel('Number of Posts')

plt.tight_layout()
plt.show()

## Part-5: Content Overview

In [ ]:
# text posts vs link posts
has_selftext = posts_df['selftext'].notna() & (posts_df['selftext'].str.strip() != '')
print(f'text posts: {has_selftext.sum():,}  ({has_selftext.mean()*100:.1f}%)')
print(f'link posts: {(~has_selftext).sum():,}  ({(~has_selftext).mean()*100:.1f}%)')

In [ ]:
# selftext length distribution (text posts only)
text_posts = posts_df[has_selftext].copy()
text_posts['selftext_words'] = text_posts['selftext'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 4))
plt.hist(text_posts['selftext_words'].clip(upper=500), bins=50, color='steelblue', edgecolor='white')
plt.title('Post Length Distribution (clipped at 500 words)', fontweight='bold')
plt.xlabel('Word Count')
plt.ylabel('Number of Posts')
plt.tight_layout()
plt.show()

print(f'median: {text_posts["selftext_words"].median():.0f} words  |  mean: {text_posts["selftext_words"].mean():.0f} words')

In [ ]:
# comment body length distribution
valid_comments = comments_df[comments_df['body'].notna()].copy()
valid_comments['body_words'] = valid_comments['body'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(10, 4))
plt.hist(valid_comments['body_words'].clip(upper=300), bins=50, color='steelblue', edgecolor='white')
plt.title('Comment Length Distribution (clipped at 300 words)', fontweight='bold')
plt.xlabel('Word Count')
plt.ylabel('Number of Comments')
plt.tight_layout()
plt.show()

print(f'median: {valid_comments["body_words"].median():.0f} words  |  mean: {valid_comments["body_words"].mean():.0f} words')

## Part-6: BG/NBD Inputs

In [ ]:
# compute BG/NBD inputs per qualifying user
bgnbd = user_stats.copy()
bgnbd['T']         = (OBS_END - bgnbd['first_post']).dt.days / 7
bgnbd['recency']   = (bgnbd['last_post'] - bgnbd['first_post']).dt.days / 7
bgnbd['frequency'] = bgnbd['n_activity'] - 1

bgnbd = bgnbd[bgnbd['T'] > 0].reset_index(drop=True)
print(f'users for BG/NBD modeling: {len(bgnbd):,}')
bgnbd[['frequency', 'recency', 'T']].head()

In [ ]:
# frequency, recency, T distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(bgnbd['frequency'].clip(upper=300), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Frequency', fontweight='bold')
axes[0].set_xlabel('Repeat transactions')
axes[0].set_ylabel('Number of Users')

axes[1].hist(bgnbd['recency'], bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('Recency (weeks)', fontweight='bold')
axes[1].set_xlabel('First to last activity')

axes[2].hist(bgnbd['T'], bins=50, color='steelblue', edgecolor='white')
axes[2].set_title('T / Tenure (weeks)', fontweight='bold')
axes[2].set_xlabel('First activity to Jan 2026')

plt.tight_layout()
plt.show()

In [ ]:
# pairwise scatter plots (sample 5000 users for readability)
sample = bgnbd.sample(min(5000, len(bgnbd)), random_state=42)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].scatter(sample['T'], sample['frequency'], alpha=0.2, s=5, color='steelblue')
axes[0].set_xlabel('T (weeks)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('T vs Frequency', fontweight='bold')

axes[1].scatter(sample['T'], sample['recency'], alpha=0.2, s=5, color='steelblue')
axes[1].set_xlabel('T (weeks)')
axes[1].set_ylabel('Recency (weeks)')
axes[1].set_title('T vs Recency', fontweight='bold')

axes[2].scatter(sample['recency'], sample['frequency'], alpha=0.2, s=5, color='steelblue')
axes[2].set_xlabel('Recency (weeks)')
axes[2].set_ylabel('Frequency')
axes[2].set_title('Recency vs Frequency', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# summary stats
bgnbd[['frequency', 'recency', 'T']].describe().round(2)

## Part-7: Network Preview

In [ ]:
# build reply edges from parent_id
# 't1_' prefix means reply to a comment; strip prefix to get parent comment's base id
comment_author = comments_df.set_index('id')['author'].to_dict()

replies = comments_df[comments_df['parent_id'].str.startswith('t1_', na=False)].copy()
replies['parent_comment_id'] = replies['parent_id'].str[3:]
replies['parent_author']     = replies['parent_comment_id'].map(comment_author)

edges_df = replies[['author', 'parent_author']].dropna()
edges_df = edges_df[edges_df['author'] != edges_df['parent_author']]

print(f'total reply edges:             {len(edges_df):,}')
print(f'unique users in reply network: {pd.concat([edges_df["author"], edges_df["parent_author"]]).nunique():,}')

In [ ]:
# in-degree and out-degree distributions
out_deg = edges_df['author'].value_counts()
in_deg  = edges_df['parent_author'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(out_deg.clip(upper=500), bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Out-degree (replies sent)', fontweight='bold')
axes[0].set_xlabel('Out-degree')
axes[0].set_ylabel('Number of Users')

axes[1].hist(in_deg.clip(upper=500), bins=50, color='steelblue', edgecolor='white')
axes[1].set_title('In-degree (replies received)', fontweight='bold')
axes[1].set_xlabel('In-degree')

plt.tight_layout()
plt.show()

In [ ]:
# log-log degree distribution (power-law check)
out_deg_counts = out_deg.value_counts().sort_index()
in_deg_counts  = in_deg.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].scatter(out_deg_counts.index, out_deg_counts.values, alpha=0.5, s=8, color='steelblue')
axes[0].set_xscale('log')
axes[0].set_yscale('log')
axes[0].set_title('Out-degree (log-log)', fontweight='bold')
axes[0].set_xlabel('Degree')
axes[0].set_ylabel('Count')

axes[1].scatter(in_deg_counts.index, in_deg_counts.values, alpha=0.5, s=8, color='steelblue')
axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_title('In-degree (log-log)', fontweight='bold')
axes[1].set_xlabel('Degree')

plt.tight_layout()
plt.show()

In [ ]:
# basic network stats
all_users = pd.concat([edges_df['author'], edges_df['parent_author']])
print(f'nodes:             {all_users.nunique():,}')
print(f'edges:             {len(edges_df):,}')
print(f'avg out-degree:    {out_deg.mean():.2f}')
print(f'median out-degree: {out_deg.median():.0f}')
print(f'max out-degree:    {out_deg.max():,}')